# 07.9 - NLP Synthesis & Review

**Phase:** 07 - NLP

**Status:** VERIFIED

---
## 1. What Are We Solving?

This is the cumulative unit: combine preprocessing, feature engineering, embeddings, sequence models, and evaluation into one end-to-end **sentiment analysis system**. We compare a classical ML baseline (TF-IDF + Logistic Regression) against a deep learning model (Embeddings + LSTM) and analyze errors.

## 2. Why Does This Matter?

Knowing individual techniques isn't enough. You must architect a full pipeline: choose preprocessing, pick representations, train, evaluate properly, and analyze failures. This mirrors real production NLP work.

## 3. Prerequisites

- All of Units 07.1-07.8

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build a classical text classification pipeline
- Build an embedding + LSTM deep pipeline in PyTorch
- Evaluate both with F1, confusion matrix, per-class metrics
- Compare approaches and conduct error analysis

## 5. Mental Model / Architecture

```text
Raw text -> preprocess -> EDA
                |
      +---------+--------+
      |                  |
  TF-IDF features    Token sequences
      |                  |
  Logistic Reg.      Embedding + LSTM
      |                  |
  Predictions        Predictions
      +--------+--------+
               |
        Evaluation & Comparison
               |
        Error Analysis
```


## 6. Setup

CPU-only; small hand-crafted sentiment corpus for fast, reproducible runs.


In [1]:
import matplotlib
matplotlib.use('Agg')
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

torch.manual_seed(0)
np.random.seed(0)
print('torch', torch.__version__)


torch 2.13.0+cpu


## 7. Load & Explore a Small Corpus

We build a sentiment corpus with positive/negative reviews, including negation and mixed cases to create real failure modes.


In [2]:
texts = [
    "i love this product it is fantastic",
    "terrible experience waste of money",
    "great value highly recommend",
    "very disappointing quality",
    "amazing quality and fast shipping",
    "not worth it avoid this",
    "best purchase i ever made",
    "poor customer service never again",
    "wonderful delightful love it",
    "horrible broken do not buy",
    "the camera is not good at night",
    "this is not a great movie unfortunately",
    "absolutely loved the experience",
    "nothing good about this at all",
    "would never return to this place",
    "fantastic service and great staff",
]
labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1]  # 1 = positive

print("Docs:", len(texts))
print("Label distribution:", Counter(labels))
print("\nAvg length:", round(np.mean([len(t.split()) for t in texts]), 1), "words")
print("Vocab size (raw):", len(set(' '.join(texts).split())))


Docs: 16
Label distribution: Counter({0: 9, 1: 7})

Avg length: 5.2 words
Vocab size (raw): 59


## 8. Preprocessing Pipeline

Negation-safe: keep 'not', 'never', 'no', 'do' so the model can learn contrast.


In [3]:
STOP = set("the a an is are was were be been being this that these those and or but for to of in on at by with from as it its them they we you i my me he she his her our us their there then than so such can could would should".split())
KEEP = {'not', 'no', 'never', 'nor', 'none'}

def preprocess(t):
    t = re.sub(r"<[^>]+>", "", t)
    t = re.sub(r"[^a-zA-Z\s]", " ", t)
    toks = t.lower().split()
    return [x for x in toks if x in KEEP or x not in STOP]

clean_texts = [preprocess(t) for t in texts]
print("Sample preprocessed:")
for t, c in zip(texts, clean_texts):
    print(f"  {t[:28]:30s} -> {c}")
    if len([1]) > 3:
        break


Sample preprocessed:
  i love this product it is fa   -> ['love', 'product', 'fantastic']
  terrible experience waste of   -> ['terrible', 'experience', 'waste', 'money']
  great value highly recommend   -> ['great', 'value', 'highly', 'recommend']
  very disappointing quality     -> ['very', 'disappointing', 'quality']
  amazing quality and fast shi   -> ['amazing', 'quality', 'fast', 'shipping']
  not worth it avoid this        -> ['not', 'worth', 'avoid']
  best purchase i ever made      -> ['best', 'purchase', 'ever', 'made']
  poor customer service never    -> ['poor', 'customer', 'service', 'never', 'again']
  wonderful delightful love it   -> ['wonderful', 'delightful', 'love']
  horrible broken do not buy     -> ['horrible', 'broken', 'do', 'not', 'buy']
  the camera is not good at ni   -> ['camera', 'not', 'good', 'night']
  this is not a great movie un   -> ['not', 'great', 'movie', 'unfortunately']
  absolutely loved the experie   -> ['absolutely', 'loved', 'experience']
  n

## 9. Split (Stratified)


In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.25,
                                           random_state=42, stratify=labels)
print("Train:", len(X_tr), "Test:", len(X_te))
print("Test labels:", y_te)


Train: 12 Test: 4
Test labels: [0, 1, 0, 1]


## 10. Model A: Classical TF-IDF + Logistic Regression


In [5]:
pipe = Pipeline([
    # vectorizer receives raw text; sklearn applies its own tokenizer/lowercase
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=2000, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
pipe.fit(X_tr, y_tr)
y_pred_cl = pipe.predict(X_te)
print("F1 (binary):", round(f1_score(y_te, y_pred_cl), 3))
print("\nClassification report:")
print(classification_report(y_te, y_pred_cl, zero_division=0))


F1 (binary): 0.5

Classification report:
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.50      0.50      0.50         2

    accuracy                           0.50         4
   macro avg       0.50      0.50      0.50         4
weighted avg       0.50      0.50      0.50         4



## 11. Build Tokens/Vocab for the LSTM

Encode preprocessed tokens into integer sequences padded to a fixed length.


In [6]:
MAXLEN = 12

def build_vocab(clean_list):
    c = Counter(w for doc in clean_list for w in doc)
    vocab = {w: i+2 for i, (w, _) in enumerate(c.most_common())}  # 0=pad,1=unk
    return vocab

# Build vocab on TRAIN tokens only (avoid leakage)
tr_clean = [preprocess(t) for t in X_tr]
te_clean = [preprocess(t) for t in X_te]
vocab = build_vocab(tr_clean)
print("Vocab size:", len(vocab))

def encode(doc, maxlen):
    ids = [vocab.get(w, 1) for w in doc][:maxlen]
    ids = ids + [0]*(maxlen-len(ids))
    return ids

Xtr_seq = torch.tensor([encode(d, MAXLEN) for d in tr_clean], dtype=torch.long)
Xte_seq = torch.tensor([encode(d, MAXLEN) for d in te_clean], dtype=torch.long)
ytr = torch.tensor(y_tr, dtype=torch.long)
yte = torch.tensor(y_te, dtype=torch.long)
print("LSTM input shapes:", Xtr_seq.shape, Xte_seq.shape)


Vocab size: 38
LSTM input shapes: torch.Size([12, 12]) torch.Size([4, 12])


## 12. Model B: Embedding + LSTM


In [7]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, 2)
    def forward(self, x):
        emb = self.embedding(x)
        out, (h, c) = self.lstm(emb)
        hcat = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(hcat)

lstm = SentimentLSTM(len(vocab)+2)
print(lstm)


SentimentLSTM(
  (embedding): Embedding(40, 32, padding_idx=0)
  (lstm): LSTM(32, 32, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


## 13. Train the LSTM (small, CPU-friendly)


In [8]:
opt = torch.optim.Adam(lstm.parameters(), lr=0.01)
lossfn = nn.CrossEntropyLoss()
Xfull = torch.cat([Xtr_seq, Xte_seq])
yfull = torch.cat([ytr, yte])

epochs = 300
for ep in range(epochs):
    opt.zero_grad()
    logits = lstm(Xtr_seq)
    loss = lossfn(logits, ytr)
    loss.backward()
    nn.utils.clip_grad_norm_(lstm.parameters(), 1.0)
    opt.step()
    if ep % 75 == 0:
        print(f"epoch {ep:3d}: loss={loss.item():.3f}")

lstm.eval()
with torch.no_grad():
    y_pred_lstm = lstm(Xte_seq).argmax(1).numpy()
    tr_acc = (lstm(Xtr_seq).argmax(1) == ytr).float().mean().item()
print("\nTrain accuracy:", round(tr_acc, 3))


epoch   0: loss=0.685


epoch  75: loss=0.000


epoch 150: loss=0.000


epoch 225: loss=0.000



Train accuracy: 1.0


## 14. Evaluation & Comparison


In [9]:
print("="*50)
print("CLASSICAL (TF-IDF + LR)")
print(classification_report(y_te, y_pred_cl, target_names=['neg','pos'], zero_division=0))
print("Confusion matrix (rows=actual, cols=pred):")
print(confusion_matrix(y_te, y_pred_cl))

print("="*50)
print("DEEP (Embedding + BiLSTM)")
print(classification_report(y_te, y_pred_lstm, target_names=['neg','pos'], zero_division=0))
print("Confusion matrix (rows=actual, cols=pred):")
print(confusion_matrix(y_te, y_pred_lstm))

f1_cl = f1_score(y_te, y_pred_cl, zero_division=0)
f1_dl = f1_score(y_te, y_pred_lstm, zero_division=0)
print(f"\nF1 classical: {f1_cl:.3f} | F1 LSTM: {f1_dl:.3f}")


CLASSICAL (TF-IDF + LR)
              precision    recall  f1-score   support

         neg       0.50      0.50      0.50         2
         pos       0.50      0.50      0.50         2

    accuracy                           0.50         4
   macro avg       0.50      0.50      0.50         4
weighted avg       0.50      0.50      0.50         4

Confusion matrix (rows=actual, cols=pred):
[[1 1]
 [1 1]]
DEEP (Embedding + BiLSTM)
              precision    recall  f1-score   support

         neg       0.50      1.00      0.67         2
         pos       0.00      0.00      0.00         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4

Confusion matrix (rows=actual, cols=pred):
[[2 0]
 [2 0]]

F1 classical: 0.500 | F1 LSTM: 0.000


## 15. Error Analysis

Identify which test reviews each model gets wrong — the crux of real-world NLP.


In [10]:
print(f"{'Text':40s} {'TRUE':6s} {'LR':4s} {'LSTM':5s}")
for i in range(len(X_te)):
    mark_cl = "*" if y_pred_cl[i] != y_te[i] else " "
    mark_dl = "*" if y_pred_lstm[i] != y_te[i] else " "
    print(f"{X_te[i][:38]:40s} {y_te[i]:<6d} {y_pred_cl[i]}{mark_cl:2s}  {y_pred_lstm[i]}{mark_dl}")

print("\n* = misclassified. Negation-heavy reviews ('not good') are the classic failure front.")


Text                                     TRUE   LR   LSTM 
poor customer service never again        0      0    0 
fantastic service and great staff        1      1    0*
this is not a great movie unfortunatel   0      1*   0 
amazing quality and fast shipping        1      0*   0*

* = misclassified. Negation-heavy reviews ('not good') are the classic failure front.


## 16. Failure Case: Sarcasm & Negation

Both models struggle when the literal words are positive but meaning is negative.


In [11]:
tricky = [
    "great job breaking everything again",
    "wow what an amazing waste of time",
    "not bad at all actually pretty good",
]

def predict_both(text):
    cl = pipe.predict([text])[0]
    seq = torch.tensor([encode(preprocess(text), MAXLEN)], dtype=torch.long)
    lstm.eval()
    with torch.no_grad():
        dl = lstm(seq).argmax(1).item()
    return cl, dl

for t in tricky:
    cl, dl = predict_both(t)
    print(f"{t:40s} LR={cl} LSTM={dl}   (pos=1,neg=0)")
print("\nSarcasm requires context beyond word count — a known limits of classical + simple deep models.")


great job breaking everything again      LR=1 LSTM=0   (pos=1,neg=0)


wow what an amazing waste of time        LR=0 LSTM=0   (pos=1,neg=0)


not bad at all actually pretty good      LR=0 LSTM=0   (pos=1,neg=0)

Sarcasm requires context beyond word count — a known limits of classical + simple deep models.


## 17. Debugging: Common Errors

- **Leakage** — building vocab/vectorizer on test. Fix: build on train only.
- **Imbalance** — use class_weight / balanced, report F1 not accuracy.
- **Pad tokens leaking** — masked padding hardens LSTM training.
- **Tiny corpus** — high variance; NLP needs much more data in practice.

## 18. Real-World Considerations

- Classical is fast/interpretable; deep captures order. Often start classical, add deep for SOTA.
- Save the pipeline + vocab + model for consistent inference.
- Monitor for drift after deployment; re-evaluate on held-out data.

## 19. Common Mistakes

- Fitting vectorizer on full data before splitting.
- Reporting only accuracy.
- No error analysis — you can't improve what you don't inspect.

## 20. When NOT to Use

- Deep models for tiny data or when interpretability is mandatory.
- Classical features when you need semantic meaning / long-range context.

## 21. Challenge

Visualize the attention weights on the LSTM to see which tokens drive each prediction (see 07.7).


In [12]:
# Challenge: inspect which tokens the embedding attends to (simple saliency)
def saliency(text):
    ids = torch.tensor([encode(preprocess(text), MAXLEN)], dtype=torch.long, requires_grad=False)
    emb = lstm.embedding(ids)
    # weight of each position by the norm of its embedding
    norms = emb.squeeze(0).norm(dim=1).detach().numpy()
    words = preprocess(text)[:MAXLEN]
    return list(zip(words, [round(float(n),2) for n in norms]))

for t in ["absolutely loved the experience", "horrible broken do not buy"]:
    print(t, "->", saliency(t)[:8])
print("\nEmbedding magnitudes give a rough sense of which tokens are salient.")


absolutely loved the experience -> [('absolutely', 5.11), ('loved', 6.75), ('experience', 4.95)]
horrible broken do not buy -> [('horrible', 5.76), ('broken', 6.25), ('do', 6.15), ('not', 6.17), ('buy', 5.59)]

Embedding magnitudes give a rough sense of which tokens are salient.


## 22. Knowledge Check

- Why did you choose these preprocessing steps?
- Which model performs better here and why?
- What are the failure modes of each approach?
- How would you deploy and monitor this system?

## 23. Teach-Back Questions

- Compare classical vs deep NLP approaches with justification.
- Walk through a full error analysis and what you'd improve next.

## 24. Summary

You built an end-to-end sentiment system with a classical baseline and an embedding+LSTM model, evaluated both with proper metrics, and performed error analysis. This is the full NLP workflow in one notebook.

## 25. Further Experiment

- Scale to a real dataset (IMDB) with acceleration.
- Add a Transformer (Phase 08) as a third model.
- Tune hyperparameters jointly and report confidence intervals.

## 26. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, torch, scikit-learn, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
